# 12 · Modelo panel — GradientBoosting global

Un unico GradientBoosting entrenado sobre **todos los municipios a la vez** (panel), con one-hot de municipio + las mismas features que 11.

- Ventaja: comparte estructura entre municipios (mas datos, menos overfitting); desventaja: pierde efectos especificos
- Mismo split y prediccion recursiva que 11
- Resultados: `results/12_modelo_panel_gbm_metrics.csv`


In [ ]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
DATA = Path("data")
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

# --- Datos
abast = pl.read_csv(DATA / "abastecimiento_urbano_baleares.csv", infer_schema_length=None).select(["cod_municipio", "anio", "consumo_hm3"])
presion = pl.read_csv(DATA / "presion_humana.csv", infer_schema_length=None)
ocup = pl.read_csv(DATA / "ocupacion_turistica.csv", infer_schema_length=None)
lluvia = pl.read_csv(DATA / "lluvia_masa_subterranea.csv", infer_schema_length=None)
mma = pl.read_csv(DATA / "municipio_masa_subterranea.csv", infer_schema_length=None)
mun = pl.read_csv(DATA / "municipio.csv", infer_schema_length=None).select(["cod_municipio", "cod_provincia", "nombre_municipio"])

# Isla IPH: 071 (Formentera) y 072 (Eivissa) comparten serie NUTS
isla_map = pl.DataFrame({
    "cod_provincia": [71, 72, 73, 74],
    "isla": ["Eivissa i Formentera", "Eivissa i Formentera", "Mallorca", "Menorca"],
})

# --- Features
iph = (presion.group_by(["nombre_isla", "anio"])
       .agg(iph_media=pl.col("iph").mean(), iph_max=pl.col("iph").max()))
ocup_m = (ocup.group_by(["cod_municipio_ine", "anio"])
          .agg(ocupacion_media=pl.col("ocupacion_plazas_pct").mean()))
ll_m = (lluvia.group_by(["cod_masa", "anio"])
        .agg(lluvia_anual_mm=pl.col("precipitacion_mm").sum())
        .join(mma, on="cod_masa")
        .group_by(["cod_municipio", "anio"])
        .agg(lluvia_anual_mm=pl.col("lluvia_anual_mm").mean()))

panel = (
    abast
    .join(mun, on="cod_municipio", how="left")
    .join(isla_map, on="cod_provincia", how="left")
    .join(iph, left_on=["isla", "anio"], right_on=["nombre_isla", "anio"], how="left")
    .join(ocup_m, left_on=["cod_municipio", "anio"], right_on=["cod_municipio_ine", "anio"], how="left")
    .join(ll_m, on=["cod_municipio", "anio"], how="left")
    .select([
        "cod_municipio", "nombre_municipio", "isla", "anio", "consumo_hm3",
        "iph_media", "iph_max", "ocupacion_media", "lluvia_anual_mm",
    ])
    .with_columns(
        pl.col("ocupacion_media").fill_null(0.0),
        pl.col("lluvia_anual_mm").fill_null(pl.col("lluvia_anual_mm").mean()),
    )
)

TEST_START = 2022
FEATURES = ["anio", "iph_media", "iph_max", "ocupacion_media", "lluvia_anual_mm", "lag1"]

panel = (
    panel
    .filter(pl.col("anio") >= 2015)
    .sort(["cod_municipio", "anio"])
    .with_columns(lag1=pl.col("consumo_hm3").shift(1).over("cod_municipio"))
)
print("panel:", panel.shape)
panel.head(8)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metricas(test, pred):
    test, pred = np.asarray(test, float), np.asarray(pred, float)
    return dict(
        mae=float(mean_absolute_error(test, pred)),
        mape=float(np.mean(np.abs((test - pred) / test)) * 100),
        rmse=float(mean_squared_error(test, pred) ** 0.5),
        r2=float(r2_score(test, pred)),
    )

def particiones(g):
    g = g.sort("anio")
    train = g.filter(pl.col("anio") < TEST_START)
    test = g.filter(pl.col("anio") >= TEST_START)
    return train, test


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# One-hot del municipio + features
pdf = panel.to_pandas()
dummies = pd.get_dummies(pdf["cod_municipio"], prefix="mun", dtype=float)
X_all = pd.concat([pdf[FEATURES], dummies], axis=1)
X_all["cod_municipio"] = pdf["cod_municipio"].values
X_all["consumo_hm3"] = pdf["consumo_hm3"].values
feature_cols = [c for c in X_all.columns if c not in ("cod_municipio", "consumo_hm3")]
print("features:", len(feature_cols))


In [ ]:
train = X_all[X_all["anio"] < TEST_START].dropna(subset=["lag1"])
test = X_all[X_all["anio"] >= TEST_START]

model = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
model.fit(train[feature_cols], train["consumo_hm3"])

def predict_recursivo_panel(model, train_rows, test_rows):
    last = float(train_rows["consumo_hm3"].iloc[-1])
    preds = []
    for _, row in test_rows.iterrows():
        x = row[feature_cols].astype(float).to_dict()
        x["lag1"] = last
        p = max(float(model.predict([list(x[f] for f in feature_cols)])[0]), 0.0)
        preds.append(p)
        last = p
    return preds

filas = []
for cod_t, g in panel.partition_by("cod_municipio", as_dict=True).items():
    cod = int(cod_t[0])
    tr, te = particiones(g)
    tr = tr.drop_nulls(subset=["lag1"])
    tr_p = X_all[(X_all["cod_municipio"] == cod) & (X_all["anio"] < TEST_START) & (X_all["lag1"].notna())]
    te_p = X_all[(X_all["cod_municipio"] == cod) & (X_all["anio"] >= TEST_START)]
    pred = predict_recursivo_panel(model, tr_p, te_p)
    filas.append({"modelo": "gb_panel", "cod_municipio": cod,
                  **metricas(te["consumo_hm3"].to_list(), pred)})

res = pd.DataFrame(filas)
res.to_csv(RESULTS / "12_modelo_panel_gbm_metrics.csv", index=False)
print(res[["mae", "mape", "rmse", "r2"]].mean().round(3))


In [ ]:
# ── Comparacion con naive y con el modelo por municipio
base = pd.read_csv(RESULTS / "10_baseline_metrics.csv")
naive = base[base["modelo"] == "naive"].set_index("cod_municipio")["mape"]
m11 = pd.read_csv(RESULTS / "11_modelo_municipio_metrics.csv").set_index("cod_municipio")["mape"]

cmp = res.set_index("cod_municipio")["mape"].to_frame("mape_gb_panel")
cmp["mape_naive"] = naive
cmp["mape_gb_municipio"] = m11
print("mejora sobre naive:", (cmp["mape_gb_panel"] < cmp["mape_naive"]).sum(), "de", cmp.shape[0])
print("mejora sobre gb_municipio:", (cmp["mape_gb_panel"] < cmp["mape_gb_municipio"]).sum(), "de", cmp.shape[0])
cmp.mean().round(2)


**Conclusiones**

- El panel comparte senal entre municipios (one-hot) y tiene ~6x mas filas de entrenamiento que un modelo por municipio.
- Suele generalizar mejor en municipios pequenos; en los grandes el one-hot apenas ayuda frente al modelo individual.
